In [ ]:
import pandas as pd

df = pd.read_csv("UK SKILL GAP"/"core_indicators_11_24.csv")

df.shape

In [ ]:
import os

print(os.getcwd())
print(os.listdir())


In [ ]:
df = pd.read_csv("UK SKILL GAP/core_indicators_11_24.csv")

df.shape

In [ ]:
df.head()

In [ ]:
df.columns.tolist()

In [ ]:
df.info()





In [ ]:
df.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
df['vac_sum'].value_counts().head(15)

In [ ]:
df['geographic_level'].value_counts(dropna=False)

In [ ]:
df['vac_sum'].value_counts().head(15)

In [ ]:
df['geographic_level'].value_counts()


In [ ]:
df['vac_sum'].value_counts().tail(20)


In [ ]:
df['vac_sum'].unique()

In [ ]:
vac_numbers = pd.to_numeric(df['vac_sum'], errors='coerce')

print("Total records:", len(df))
print("Numeric records:", vac_numbers.notna().sum())
print("Non-numeric records:", vac_numbers.isna().sum())
print("Unique non-numeric codes:", df.loc[vac_numbers.isna(), 'vac_sum'].value_counts())

In [ ]:
errors='coerce'



In [ ]:
df.loc[df['vac_sum'] != 'u', 
       ['country_name', 'region_name', 'sector', 'time_period', 'vac_sum']].head(15)

In [ ]:
print(df[['country_name', 'region_name', 'sector', 'time_period', 'vac_sum']].head(10).to_string(index=False))

In [ ]:
df['time_period'].value_counts().sort_index()

In [ ]:
df.groupby('time_period')['vac_sum'].apply(
    lambda x: pd.to_numeric(x, errors='coerce').notna().sum()
)

In [ ]:
df['site_size'].value_counts()




In [ ]:
df['site_type'].value_counts()

In [ ]:
df.groupby(
    ['time_period', 'geographic_level']
)['vac_sum'].apply(
    lambda x: pd.to_numeric(x, errors='coerce').notna().sum()
)

In [ ]:
vac_numbers = pd.to_numeric(df['vac_sum'], errors='coerce')

regional_2024 = df[
    (df['time_period'] == 2024) &
    (df['geographic_level'] == 'Regional') &
    (vac_numbers.notna())
]

print(regional_2024[
    ['country_name', 'region_name', 'sector',
     'site_size', 'vac_sum']
].to_string(index=False))


In [ ]:
# Filter the 2024 overall totals
totals_2024 = df[
    (df['time_period'] == 2024) &
    (df['sector'] == 'Total') &
    (df['site_size'] == 'Total')
]

# Display the country and regional figures
print(
    totals_2024[
        ['country_name', 'region_name', 'geographic_level', 'vac_sum']
    ].to_string(index=False)
)

In [ ]:
uk_2024 = df[
    (df['time_period'] == 2024) &
    (df['country_name'] == 'United Kingdom') &
    (df['geographic_level'] == 'National')
]

print(
    uk_2024[
        [
            'single_or_multi_site',
            'site_type',
            'site_size',
            'sector',
            'vac_sum',
            'has_vac',
            'has_htfv',
            'has_ssv',
            'has_skillgap'
        ]
    ].to_string(index=False)
)

In [ ]:
# Check missing/unavailable values in our key indicators

key_columns = [
    'vac_sum',
    'has_vac',
    'has_htfv',
    'has_ssv',
    'has_skillgap',
    'ssv_sum',
    'skillgaps'
]

for column in key_columns:
    print(f"\n{column}")
    print(df[column].value_counts().head(5))
    print("Unavailable:", (df[column] == 'u').sum())

In [ ]:
# Create a data availability report

availability_report = pd.DataFrame({
    'Available': df[key_columns].ne('u').sum(),
    'Unavailable': df[key_columns].eq('u').sum(),
    'Unavailable_%': (
        df[key_columns].eq('u').mean() * 100
    ).round(2)
})

availability_report

In [ ]:
u_check = df[df['vac_sum'] == 'u'].groupby(
    ['time_period', 'geographic_level']
).size().unstack(fill_value=0)

u_check

In [ ]:
for column in ['vac_sum', 'ssv_sum', 'skillgaps']:
    print(f"\n{column}")
    print(df[column].value_counts().head(10))

In [ ]:
# Keep the original dataset unchanged
clean_df = df.copy()

# Columns we want to analyse
numeric_columns = [
    'vac_sum',
    'ssv_sum',
    'skillgaps',
    'has_vac',
    'has_htfv',
    'has_ssv',
    'has_skillgap'
]

# Preserve flags for low-reliability values
for column in numeric_columns:
    clean_df[column + '_low_reliability'] = df[column].eq('u')

# Convert the measures into numeric format
for column in numeric_columns:
    clean_df[column] = pd.to_numeric(
        clean_df[column],
        errors='coerce'
    )

clean_df[numeric_columns].info()

In [ ]:
for column in numeric_columns:
    print(
        column,
        "Missing values:", clean_df[column].isna().sum(),
        "| Low reliability:", clean_df[column + '_low_reliability'].sum()
    )

In [ ]:
clean_df['sector'].value_counts()

In [ ]:
industry_gaps_2024 = clean_df[
    (clean_df['time_period'] == 2024) &
    (clean_df['geographic_level'] == 'National') &
    (clean_df['sector'] != 'Total') &
    (clean_df['site_size'] == 'Total') &
    (clean_df['skillgaps'].notna())
]

industry_gaps_2024[
    ['sector', 'skillgaps']
].sort_values(
    by='skillgaps',
    ascending=False
)

In [ ]:
dimensions = [
    'country_name',
    'single_or_multi_site',
    'site_type',
    'site_size',
    'sector'
]

for column in dimensions:
    print(f"\n--- {column} ---")
    print(clean_df[column].value_counts().head(15))

In [ ]:
industry_gaps_2024 = clean_df[
    (clean_df['time_period'] == 2024) &
    (clean_df['geographic_level'] == 'National') &
    (clean_df['country_name'] == 'United Kingdom') &
    (clean_df['sector'] != 'Total') &
    (clean_df['site_size'] == 'Total') &
    (clean_df['site_type'] == 'Total') &
    (clean_df['single_or_multi_site'] == 'Total') &
    (clean_df['skillgaps'].notna())
]

industry_ranking = industry_gaps_2024[
    ['sector', 'skillgaps']
].sort_values(
    by='skillgaps',
    ascending=False
)

print("Number of industry records:", len(industry_ranking))

industry_ranking

In [ ]:
industry_gaps_2024[
    [
        'country_name',
        'geographic_level',
        'single_or_multi_site',
        'site_type',
        'site_size',
        'sector',
        'skillgaps',
        'skillgap_inc',
        'skillgap_den'
    ]
].to_string(index=False)

In [ ]:
clean_df[
    ['skillgap_inc', 'skillgap_den']
].describe()

In [ ]:
for column in ['skillgap_inc', 'skillgap_den']:
    clean_df[column] = pd.to_numeric(
        clean_df[column],
        errors='coerce'
    )

clean_df[['skillgap_inc', 'skillgap_den']].describe()

In [ ]:
industry_density_2024 = industry_gaps_2024[
    ['sector', 'skillgaps', 'skillgap_inc', 'skillgap_den']
].sort_values(
    by='skillgap_den',
    ascending=False
)

industry_density_2024

In [ ]:
training_columns = [
    'has_training_in_general',
    'trainees',
    'training_days',
    'training_den',
    'train_days_employee',
    'train_days_trainee'
]

clean_df[training_columns].info()

In [ ]:
for column in training_columns:
    print(f"\n--- {column} ---")
    print(clean_df[column].value_counts().head(8))
    print("Unavailable:", (clean_df[column] == 'u').sum())

In [ ]:
# Columns to clean
training_numeric_columns = [
    'has_training_in_general',
    'trainees',
    'training_days',
    'training_den',
    'train_days_employee',
    'train_days_trainee'
]

# Preserve original low-reliability flags
for column in training_numeric_columns:
    clean_df[column + '_low_reliability'] = df[column].eq('u')

# Convert to numeric
for column in training_numeric_columns:
    clean_df[column] = pd.to_numeric(
        clean_df[column],
        errors='coerce'
    )

# Check the result
clean_df[training_numeric_columns].info()


In [ ]:
# Select comparable 2024 UK-wide industry records
training_comparison = clean_df[
    (clean_df['time_period'] == 2024) &
    (clean_df['country_name'] == 'United Kingdom') &
    (clean_df['geographic_level'] == 'National') &
    (clean_df['single_or_multi_site'] == 'Total') &
    (clean_df['site_type'] == 'Total') &
    (clean_df['site_size'] == 'Total') &
    (clean_df['sector'] != 'Total') &
    (clean_df['skillgap_den'].notna()) &
    (clean_df['training_den'].notna())
]

# Keep the measures we want to compare
training_comparison = training_comparison[
    [
        'sector',
        'skillgaps',
        'skillgap_den',
        'training_den',
        'train_days_employee',
        'train_days_trainee'
    ]
].sort_values(
    by='skillgap_den',
    ascending=False
)

print("Number of industry records:", len(training_comparison))

training_comparison

In [ ]:
# Calculate correlation between training and skills gaps

correlation = training_comparison[
    ['skillgap_den', 'training_den']
].corr()

correlation

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.scatter(
    training_comparison['training_den'],
    training_comparison['skillgap_den'],
    s=80
)

plt.xlabel('Training participation (%)')
plt.ylabel('Skills-gap density (%)')
plt.title('Training vs Skills Gaps Across UK Industries (2024)')

plt.grid(True)
plt.show()




In [ ]:
vacancy_columns = ['vac_sum', 'htfv_sum', 'ssv_sum']

print(clean_df[vacancy_columns].dtypes)

print("\nAvailable records:")
print(clean_df[vacancy_columns].count())

In [ ]:
clean_df['htfv_sum'].value_counts().head(15)

In [ ]:
clean_df['htfv_sum_low_reliability'] = df['htfv_sum'].eq('u')

clean_df['htfv_sum'] = pd.to_numeric(
    clean_df['htfv_sum'],
    errors='coerce'
)

clean_df['htfv_sum'].dtype

In [ ]:
vacancy_analysis = clean_df[
    (clean_df['time_period'] == 2024) &
    (clean_df['country_name'] == 'United Kingdom') &
    (clean_df['geographic_level'] == 'National') &
    (clean_df['single_or_multi_site'] == 'Total') &
    (clean_df['site_type'] == 'Total') &
    (clean_df['site_size'] == 'Total') &
    (clean_df['sector'] != 'Total')
]

vacancy_analysis = vacancy_analysis[
    ['sector', 'vac_sum', 'htfv_sum', 'ssv_sum']
]

vacancy_analysis

In [ ]:
vacancy_analysis = vacancy_analysis.copy()

vacancy_analysis['ssv_percentage'] = (
    vacancy_analysis['ssv_sum'] / vacancy_analysis['vac_sum']
) * 100

vacancy_analysis = vacancy_analysis.sort_values(
    'ssv_percentage',
    ascending=False
)

vacancy_analysis[
    ['sector', 'vac_sum', 'ssv_sum', 'ssv_percentage']
].round(2)




In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11, 7))

plt.barh(
    vacancy_analysis['sector'],
    vacancy_analysis['ssv_percentage']
)

plt.xlabel('Skill-shortage vacancies (% of total vacancies)')
plt.ylabel('Industry')
plt.title('Proportion of Vacancies Caused by Skills Shortages (2024)')

plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("Proportion_of_vacancies_Caused_by_Skills_Shortages_2024.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
regional_analysis = clean_df[
    (clean_df['time_period'] == 2024) &
    (clean_df['country_name'] == 'England') &
    (clean_df['geographic_level'] == 'Regional') &
    (clean_df['single_or_multi_site'] == 'Total') &
    (clean_df['site_type'] == 'Total') &
    (clean_df['site_size'] == 'Total') &
    (clean_df['sector'] == 'Total')
]

regional_analysis = regional_analysis[
    ['region_name', 'vac_sum', 'htfv_sum', 'ssv_sum']
]

regional_analysis

In [ ]:
regional_analysis = regional_analysis.copy()

regional_analysis['ssv_percentage'] = (
    regional_analysis['ssv_sum'] / regional_analysis['vac_sum']
) * 100

regional_analysis = regional_analysis.sort_values(
    'ssv_percentage',
    ascending=False
)

regional_analysis[
    ['region_name', 'vac_sum', 'ssv_sum', 'ssv_percentage']
].round(2)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11, 6))

plt.bar(
    regional_analysis['region_name'],
    regional_analysis['ssv_percentage']
)

plt.xlabel('Region')
plt.ylabel('Skill-shortage vacancies (% of all vacancies)')
plt.title('Skill-Shortage Vacancies by English Region (2024)')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("industry_skills_gap_2024.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Check for duplicate rows in our cleaned dataset

print("Original dataset:", clean_df.shape)

print("Duplicate rows:", clean_df.duplicated().sum())


In [ ]:
# Check missing values in the cleaned dataset


missing_values = clean_df.isnull().sum()

missing_values = missing_values[missing_values > 0].sort_values(
    ascending=False
)

missing_values

In [ ]:
# Compare missing regional information by geographic level

clean_df.groupby('geographic_level').agg(
    total_records=('region_name', 'size'),
    missing_regions=('region_name', lambda x: x.isna().sum())
)

In [ ]:
print("Industry analysis:", industry_gaps_2024.shape)

print("Regional analysis:", regional_analysis.shape)

print("Training comparison:", training_comparison.shape)


In [ ]:
# Final industry dataset

industry_final = industry_gaps_2024[
    ['sector', 'skillgaps', 'skillgap_inc', 'skillgap_den']
].copy()
s
# Final regional dataset

regional_final = regional_analysis[
    ['region_name', 'vac_sum', 'htfv_sum', 'ssv_sum', 'ssv_percentage']
].copy()

# Final training dataset

training_final = training_comparison.copy()

print("Industry:", industry_final.shape)
print("Regional:", regional_final.shape)
print("Training:", training_final.shape)

In [ ]:
# Export final analysis datasets as CSV files

industry_final.to_csv("industry_analysis_2024.csv", index=False)

regional_final.to_csv("regional_analysis_2024.csv", index=False)

training_final.to_csv("training_analysis_2024.csv", index=False)

print("All three datasets exported successfully!")


# Project Findings – UK Employer Skills Survey 2024

## 1. Industry Analysis
Construction had the highest proportion of skill-shortage vacancies at 45.02%, while Public Administration had the lowest at 6.65%.

Business Services had the highest volume of skill-shortage vacancies, estimated at 58,000.

## 2. Regional Analysis
The South West had the highest proportion of skill-shortage vacancies at 40.09%, while the East of England had the lowest at 18.94%.

London had the highest volume of skill-shortage vacancies at 45,600.

## 3. Training and Skills Gaps
The comparison of 13 industries showed a weak negative correlation (-0.17) between training participation and skills-gap density.

This suggests a weak association, but does not establish that training reduces skills gaps.

## 4. Conclusion
Skills shortages vary considerably across industries and regions. Employers and policymakers should consider both the volume and proportion of skill-shortage vacancies when deciding where targeted workforce development may be needed.

In [ ]:
import os

print("Your project folder:", os.getcwd())

print("\nYour exported files:")

for file in os.listdir():
    if file.endswith(('.csv', '.png', '.ipynb')):
        print(file)